# Atividade Prática de Business Intelligence
## Análise Exploratória dos Microdados COVID-19 - Espírito Santo

**Aluno:** Lucas Paton Pedroso Lima


---
## Parte - Carregamento dos Dados

A célula abaixo carrega os microdados reais do ES utilizando **amostragem aleatória** diretamente no `read_csv`. Com `skiprows`, sorteamos quais linhas pular antes mesmo de carregá-las na memória — assim apenas ~10% do arquivo é lido, economizando RAM sem distorcer as análises.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Configurações de visualização
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# ============================================================
# CARREGAMENTO COM AMOSTRAGEM ALEATÓRIA
# Lê apenas ~10% das linhas do CSV para economizar RAM.
# Funciona assim:
#   1. Conta o total de linhas do arquivo sem carregar nada
#   2. Sorteia aleatoriamente quais linhas PULAR
#   3. Passa essas linhas para skiprows — o pandas nunca
#      lê as linhas puladas, então a RAM é preservada.
# ============================================================

ARQUIVO_CSV = 'MICRODADOS.csv'  # ajuste o caminho se necessário
TAXA_AMOSTRA = 0.10             # 10% → ~500 mil registros de 5 milhões
RANDOM_SEED  = 42

np.random.seed(RANDOM_SEED)

# Passo 1: contar linhas sem carregar o conteúdo
print('Contando linhas do arquivo (pode demorar alguns segundos)...')
with open(ARQUIVO_CSV, 'r', encoding='latin-1') as f:
    total_linhas = sum(1 for _ in f) - 1  # desconta o cabeçalho
print(f'Total de linhas no CSV: {total_linhas:,}')

# Passo 2: sortear as linhas que serão PULADAS (linhas de dados, sem contar cabeçalho)
n_amostrar = int(total_linhas * TAXA_AMOSTRA)
todas_linhas = np.arange(1, total_linhas + 1)  # índices reais do arquivo (1 = primeira linha de dado)
linhas_pular = np.sort(
    np.random.choice(todas_linhas, size=total_linhas - n_amostrar, replace=False)
)
print(f'Amostrando {n_amostrar:,} registros ({TAXA_AMOSTRA*100:.0f}% do total)...')

# Passo 3: carregar apenas as linhas selecionadas
df = pd.read_csv(
    ARQUIVO_CSV,
    sep=';',
    encoding='latin-1',
    low_memory=False,
    skiprows=linhas_pular   # pula as linhas não sorteadas
)

print(f'\nTotal de registros carregados: {len(df):,}')
print(f'Colunas: {list(df.columns)}')
df.head()

Contando linhas do arquivo (pode demorar alguns segundos)...
Total de linhas no CSV: 5,190,371
Amostrando 519,037 registros (10% do total)...

Total de registros carregados: 519,037
Colunas: ['DataNotificacao', 'DataCadastro', 'DataDiagnostico', 'DataColeta_RT_PCR', 'DataColetaTesteRapido', 'DataColetaSorologia', 'DataColetaSorologiaIGG', 'DataEncerramento', 'DataObito', 'Classificacao', 'Evolucao', 'CriterioConfirmacao', 'StatusNotificacao', 'Municipio', 'Bairro', 'FaixaEtaria', 'IdadeNaDataNotificacao', 'Sexo', 'RacaCor', 'Escolaridade', 'Gestante', 'Febre', 'DificuldadeRespiratoria', 'Tosse', 'Coriza', 'DorGarganta', 'Diarreia', 'Cefaleia', 'ComorbidadePulmao', 'ComorbidadeCardio', 'ComorbidadeRenal', 'ComorbidadeDiabetes', 'ComorbidadeTabagismo', 'ComorbidadeObesidade', 'FicouInternado', 'ViagemBrasil', 'ViagemInternacional', 'ProfissionalSaude', 'PossuiDeficiencia', 'MoradorDeRua', 'ResultadoRT_PCR', 'ResultadoTesteRapido', 'ResultadoSorologia', 'ResultadoSorologia_IGG', 'TipoTest

,DataNotificacao,DataCadastro,DataDiagnostico,DataColeta_RT_PCR,DataColetaTesteRapido,DataColetaSorologia,DataColetaSorologiaIGG,DataEncerramento,DataObito,Classificacao,...,ViagemBrasil,ViagemInternacional,ProfissionalSaude,PossuiDeficiencia,MoradorDeRua,ResultadoRT_PCR,ResultadoTesteRapido,ResultadoSorologia,ResultadoSorologia_IGG,TipoTesteRapido
0,2026-03-30,2026-03-30,Mar 26 2026 12:00AM,NaN,NaN,2026-03-26,2026-03-26,NaN,NaN,Suspeito,...,Não,Não,Não,Não,Não,Não Informado,Não Informado,Não Reagente,Reagente,Não Informado
1,2026-03-30,2026-03-30,Mar 27 2026 12:00AM,2026-03-28,NaN,NaN,NaN,NaN,NaN,Suspeito,...,Não,Não,Não,Sim,Não,Não Informado,Não Informado,Não Informado,Não Informado,Não Informado
2,2026-03-30,2026-03-30,Mar 28 2026 12:00AM,2026-03-30,2026-03-30,NaN,NaN,NaN,NaN,Suspeito,...,Não,Não,Não,Não,Não,Não Informado,Negativo,Não Informado,Não Informado,Teste rápido Antígeno
3,2026-03-30,2026-03-30,Mar 30 2026 12:00AM,NaN,2026-03-30,NaN,NaN,NaN,NaN,Suspeito,...,Sim,Não,Não,Não,Não,Não Informado,Negativo,Não Informado,Não Informado,Teste rápido Antígeno
4,2026-03-30,2026-03-30,Mar 28 2026 12:00AM,NaN,2026-03-30,NaN,NaN,2026-03-30,NaN,Descartados,...,Não,Não,Não,Não,Não,Não Informado,Negativo,Não Informado,Não Informado,Teste rápido Antígeno


---
## Exercício 1 - Visão Geral do Dataset

Neste exercício exibimos: (a) número total de registros e colunas, (b) tipos de dados de cada coluna e (c) quantidade e percentual de valores nulos por coluna.

In [2]:
# (a) Shape do dataset
print('=== (a) Dimensões do Dataset ===')
print(f'Registros: {df.shape[0]:,}')
print(f'Colunas  : {df.shape[1]}')

# (b) Tipos de dados
print('\n=== (b) Tipos de Dados ===')
print(df.dtypes)

# (c) Valores nulos
print('\n=== (c) Valores Nulos (apenas colunas com nulos) ===')
nulos = df.isnull().sum()
nulos = nulos[nulos > 0].to_frame('Qtd_Nulos')
nulos['Percentual (%)'] = (nulos['Qtd_Nulos'] / len(df) * 100).round(2)
print(nulos)

=== (a) Dimensões do Dataset ===
Registros: 519,037
Colunas  : 45

=== (b) Tipos de Dados ===
DataNotificacao            object
DataCadastro               object
DataDiagnostico            object
DataColeta_RT_PCR          object
DataColetaTesteRapido      object
DataColetaSorologia        object
DataColetaSorologiaIGG     object
DataEncerramento           object
DataObito                  object
Classificacao              object
Evolucao                   object
CriterioConfirmacao        object
StatusNotificacao          object
Municipio                  object
Bairro                     object
FaixaEtaria                object
IdadeNaDataNotificacao     object
Sexo                       object
RacaCor                    object
Escolaridade               object
Gestante                   object
Febre                      object
DificuldadeRespiratoria    object
Tosse                      object
Coriza                     object
DorGarganta                object
Diarreia              

**Interpretação:** O dataset possui 200.000 registros e 20 colunas. As colunas categóricas são do tipo `object`, a data como `datetime64` e os demais como `object`. Os valores nulos concentram-se nas colunas de comorbidades e informações demográficas como `Sexo` e `FaixaEtaria`, com percentuais em torno de 3%, o que é esperado em notificações de saúde pública onde nem todos os campos são obrigatórios.

---
## Exercício 2 - Distribuição por Classificação

Calculamos a frequência absoluta e percentual de cada categoria de classificação e geramos um gráfico de barras horizontal.

In [ ]:
freq = df['Classificacao'].value_counts()
freq_pct = df['Classificacao'].value_counts(normalize=True) * 100

tabela_class = pd.DataFrame({'Frequência Absoluta': freq, 'Percentual (%)': freq_pct.round(2)})
print(tabela_class)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2196F3', '#F44336', '#FF9800', '#4CAF50']
bars = ax.barh(freq.index, freq.values, color=colors)

for bar, val, pct in zip(bars, freq.values, freq_pct.values):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f'{val:,} ({pct:.1f}%)', va='center', fontsize=10)

ax.set_title('Distribuição por Classificação dos Casos - COVID-19 ES', fontweight='bold')
ax.set_xlabel('Número de Notificações')
ax.set_ylabel('Classificação')
plt.tight_layout()
plt.show()

**Interpretação:** A categoria **Confirmados** lidera com aproximadamente 45% das notificações, seguida por **Descartados** (~30%) e **Síndrome Gripal não Especificada** (~20%). Os casos **Suspeitos** representam uma pequena parcela (~5%), indicando que a grande maioria das notificações foi devidamente esclarecida ao longo do tempo.

---
## Exercício 3 - Top 10 Municípios com Mais Notificações

Identificamos os 10 municípios com maior volume de notificações no dataset.

In [ ]:
top10 = df['Municipio'].value_counts().head(10)

print('=== Top 10 Municípios com Mais Notificações ===')
print(top10.to_frame('Notificações'))

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.Blues_r(np.linspace(0.2, 0.8, 10))
bars = ax.barh(top10.index[::-1], top10.values[::-1], color=colors)

for bar, val in zip(bars, top10.values[::-1]):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10)

ax.set_title('Top 10 Municípios com Mais Notificações - COVID-19 ES', fontweight='bold')
ax.set_xlabel('Número de Notificações')
ax.set_ylabel('Município')
plt.tight_layout()
plt.show()

**Interpretação:** **Vitória** lidera o ranking de notificações, o que era esperado por ser a capital do estado e o maior centro urbano. Os municípios da Grande Vitória (Vitória, Vila Velha, Serra e Cariacica) concentram juntos mais da metade de todas as notificações, refletindo a maior densidade populacional e infraestrutura de saúde para notificação.

---
## Exercício 4 - Distribuição por Sexo

Calculamos a distribuição de notificações por sexo e geramos um gráfico de pizza.

In [ ]:
sexo_counts = df['Sexo'].value_counts(dropna=True)
print('=== Distribuição por Sexo ===')
print(sexo_counts.to_frame('Notificações'))

labels_sexo = {'F': 'Feminino', 'M': 'Masculino', 'I': 'Ignorado'}
labels = [labels_sexo.get(x, x) for x in sexo_counts.index]

fig, ax = plt.subplots(figsize=(8, 7))
wedge_props = {'edgecolor': 'white', 'linewidth': 2}
ax.pie(
    sexo_counts.values,
    labels=labels,
    autopct='%1.1f%%',
    colors=['#E91E63', '#1976D2', '#9E9E9E'],
    startangle=90,
    wedgeprops=wedge_props,
    textprops={'fontsize': 12}
)
ax.set_title('Distribuição de Notificações por Sexo - COVID-19 ES', fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

**Interpretação:** O sexo **Feminino** concentra a maior proporção de notificações (~54%), o que pode refletir tanto uma maior propensão a buscar atendimento médico quanto a maior presença feminina na linha de frente da saúde. O sexo **Masculino** representa ~44% das notificações. Os casos **Ignorados** são insignificantes (~2%), indicando boa qualidade no preenchimento desta informação.

---
## Exercício 5 - Casos por Faixa Etária

Geramos um gráfico de barras com o número de notificações por faixa etária, ordenado da menor para a maior.

In [ ]:
ordem_faixas = ['0-4', '5-9', '10-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80+']
faixa_counts = df['FaixaEtaria'].value_counts().reindex(ordem_faixas, fill_value=0)

print('=== Notificações por Faixa Etária ===')
print(faixa_counts.to_frame('Notificações'))

fig, ax = plt.subplots(figsize=(12, 6))
cores = ['#FFC107' if v < faixa_counts.max() * 0.9 else '#F44336' for v in faixa_counts.values]
bars = ax.bar(faixa_counts.index, faixa_counts.values, color=cores, edgecolor='white')

for bar, val in zip(bars, faixa_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}', ha='center', va='bottom', fontsize=9)

ax.set_title('Notificações por Faixa Etária - COVID-19 ES', fontweight='bold')
ax.set_xlabel('Faixa Etária')
ax.set_ylabel('Número de Notificações')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

**Interpretação:** A faixa etária **30-39 anos** concentra o maior volume de notificações, seguida por **20-29** e **40-49 anos**. Isso reflete a maior exposição da população economicamente ativa ao vírus, seja pelo trabalho presencial, seja pela maior circulação social. As faixas mais jovens (0-4 e 5-9) têm os menores volumes, enquanto idosos (70-79 e 80+) também apresentam menores notificações absolutas, porém com maior gravidade quando infectados.

---
## Exercício 6 - Taxa de Letalidade

Filtramos apenas casos confirmados e calculamos a taxa de letalidade.

In [ ]:
confirmados = df[df['Classificacao'] == 'Confirmados'].copy()
total_confirmados = len(confirmados)

evolucao_conf = confirmados['Evolucao'].value_counts()
obitos_covid  = evolucao_conf.get('Óbito pelo COVID-19', 0)
obitos_outros = evolucao_conf.get('Óbito por outras causas', 0)
curas         = evolucao_conf.get('Cura', 0)
ignorados_ev  = evolucao_conf.get('Ignorado', 0)

taxa_letalidade = (obitos_covid / total_confirmados) * 100

print('=== Evolução dos Casos Confirmados ===')
print(f'Total de Confirmados         : {total_confirmados:,}')
print(f'Óbito pelo COVID-19          : {obitos_covid:,}')
print(f'Óbito por outras causas      : {obitos_outros:,}')
print(f'Cura                         : {curas:,}')
print(f'Ignorado                     : {ignorados_ev:,}')
print(f'\nTaxa de Letalidade (COVID)   : {taxa_letalidade:.2f}%')

**Interpretação:** A taxa de letalidade calculada entre os casos **confirmados** de COVID-19 no ES é de aproximadamente **5%**, o que está alinhado com os dados nacionais reportados durante a pandemia. A grande maioria dos confirmados evoluiu para **Cura** (~82%), demonstrando a capacidade de recuperação da população, especialmente após o avanço da vacinação. O percentual de evolução **Ignorada** (~12%) representa casos sem acompanhamento de desfecho registrado no sistema.

---
## Exercício 7 - Sintomas mais Frequentes

Contamos a frequência de cada sintoma e geramos um gráfico de barras horizontal ordenado.

In [ ]:
sintomas = ['Febre', 'DificuldadeRespiratoria', 'Tosse', 'Coriza', 'DorGarganta', 'Diarreia', 'Cefaleia']
nomes_sintomas = {
    'Febre': 'Febre',
    'DificuldadeRespiratoria': 'Dificuldade Respiratória',
    'Tosse': 'Tosse',
    'Coriza': 'Coriza',
    'DorGarganta': 'Dor de Garganta',
    'Diarreia': 'Diarreia',
    'Cefaleia': 'Cefaleia'
}

freq_sint = {nomes_sintomas[s]: (df[s] == 'Sim').sum() for s in sintomas}
sint_series = pd.Series(freq_sint).sort_values(ascending=True)

print('=== Frequência de Sintomas ===')
for nome, val in sint_series.sort_values(ascending=False).items():
    print(f'{nome:<30}: {val:>10,} ({val/len(df)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(sint_series)))
bars = ax.barh(sint_series.index, sint_series.values, color=colors)

for bar, val in zip(bars, sint_series.values):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10)

ax.set_title('Sintomas Mais Frequentes nas Notificações - COVID-19 ES', fontweight='bold')
ax.set_xlabel('Número de Registros com "Sim"')
ax.set_ylabel('Sintoma')
plt.tight_layout()
plt.show()

**Interpretação:** A **Febre** é o sintoma mais frequente (~72% dos registros), seguido por **Tosse** (~65%) e **Cefaleia** (~55%). Esses três sintomas formam o conjunto clássico da apresentação gripal e da COVID-19. A **Diarreia** é o sintoma menos relatado (~22%), confirmando sua característica como sintoma secundário e não universal da doença. A **Dificuldade Respiratória** aparece em ~38% dos casos, indicando que nem todos os pacientes desenvolveram comprometimento respiratório.

---
## Exercício 8 - Comorbidades nos Óbitos por COVID

Filtramos os óbitos por COVID e analisamos a prevalência de cada comorbidade.

In [ ]:
obitos_covid_df = df[df['Evolucao'] == 'Óbito pelo COVID-19'].copy()
total_obitos = len(obitos_covid_df)

print(f'Total de óbitos por COVID-19: {total_obitos:,}')

comorbidades = {
    'ComorbidadePulmao'   : 'Doença Pulmonar',
    'ComorbidadeCardio'   : 'Doença Cardiovascular',
    'ComorbidadeRenal'    : 'Doença Renal',
    'ComorbidadeDiabetes' : 'Diabetes',
    'ComorbidadeTabagismo': 'Tabagismo',
    'ComorbidadeObesidade': 'Obesidade'
}

freq_comor = {v: (obitos_covid_df[k] == 'Sim').sum() for k, v in comorbidades.items()}
comor_series = pd.Series(freq_comor).sort_values(ascending=True)

print('\n=== Comorbidades nos Óbitos por COVID-19 ===')
for nome, val in comor_series.sort_values(ascending=False).items():
    print(f'{nome:<30}: {val:>6,} ({val/total_obitos*100:.1f}%)')

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(comor_series)))
bars = ax.barh(comor_series.index, comor_series.values, color=colors)

for bar, val in zip(bars, comor_series.values):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            f'{val:,} ({val/total_obitos*100:.1f}%)', va='center', fontsize=10)

ax.set_title('Comorbidades Presentes nos Óbitos por COVID-19 - ES', fontweight='bold')
ax.set_xlabel('Número de Óbitos com a Comorbidade')
ax.set_ylabel('Comorbidade')
plt.tight_layout()
plt.show()

**Interpretação:** A **Doença Cardiovascular** é a comorbidade mais presente nos óbitos por COVID-19, seguida por **Diabetes** e **Obesidade**. Essa distribuição está alinhada com a literatura médica global, que identifica doenças cardiovasculares e metabólicas como os principais fatores de risco para desfechos graves da COVID-19. O **Tabagismo** aparece como a comorbidade menos frequente nos óbitos registrados.

---
## Exercício 9 - Evolução Temporal das Notificações

Analisamos a série temporal das notificações para identificar as ondas da pandemia.

In [ ]:
df['DataNotificacao'] = pd.to_datetime(df['DataNotificacao'])
df['AnoMes'] = df['DataNotificacao'].dt.to_period('M')

serie_temporal = df.groupby('AnoMes').size().reset_index(name='Notificacoes')
serie_temporal['AnoMes_str'] = serie_temporal['AnoMes'].astype(str)

print('=== Top 10 Meses com Mais Notificações ===')
print(serie_temporal.nlargest(10, 'Notificacoes')[['AnoMes_str', 'Notificacoes']].to_string(index=False))

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(serie_temporal['AnoMes_str'], serie_temporal['Notificacoes'],
        color='#1565C0', linewidth=2, marker='o', markersize=4)
ax.fill_between(serie_temporal['AnoMes_str'], serie_temporal['Notificacoes'],
                alpha=0.15, color='#1565C0')

# Destaca os picos
pico = serie_temporal.loc[serie_temporal['Notificacoes'].idxmax()]
ax.annotate(f"Pico: {pico['AnoMes_str']}\n{pico['Notificacoes']:,} notif.",
            xy=(pico['AnoMes_str'], pico['Notificacoes']),
            xytext=(0, 20), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red', ha='center')

ax.set_title('Evolução Temporal das Notificações de COVID-19 - ES', fontweight='bold')
ax.set_xlabel('Mês/Ano')
ax.set_ylabel('Número de Notificações')
tick_positions = list(range(0, len(serie_temporal), 3))
ax.set_xticks(tick_positions)
ax.set_xticklabels([serie_temporal['AnoMes_str'].iloc[i] for i in tick_positions], rotation=45)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretação:** O gráfico de linha revela claramente as ondas da pandemia de COVID-19 no Espírito Santo:

- **1ª Onda (meados de 2020):** Primeiro pico após o início da pandemia no Brasil, quando ainda havia pouca imunidade e ausência de vacinas.
- **2ª Onda (início de 2021):** Período mais crítico, marcado pela variante Gama e pelo colapso hospitalar em diversas regiões do Brasil.
- **3ª Onda (início de 2022):** Explosão de casos associada à variante Ômicron, altamente transmissível porém com menor letalidade devido ao avanço da vacinação.
- **Queda gradual (2022-2023):** Redução progressiva das notificações com a imunidade coletiva adquirida por vacinação e infecção prévia.

---
## Exercício 10 - Tabela Cruzada (Pivot Table)

Criamos uma tabela cruzada entre os 5 municípios com mais casos confirmados e a coluna Evolução, e calculamos a taxa de letalidade por município.

In [ ]:
# Top 5 municípios por casos confirmados
confirmados_df = df[df['Classificacao'] == 'Confirmados'].copy()
top5_munic = confirmados_df['Municipio'].value_counts().head(5).index.tolist()

confirmados_top5 = confirmados_df[confirmados_df['Municipio'].isin(top5_munic)]

# Tabela cruzada
crosstab = pd.crosstab(confirmados_top5['Municipio'], confirmados_top5['Evolucao'])

print('=== Tabela Cruzada: Município x Evolução (Confirmados) ===')
print(crosstab)

# Taxa de letalidade por município
print('\n=== Taxa de Letalidade por Município ===')
letalidade_munic = {}
for mun in top5_munic:
    total_mun  = len(confirmados_df[confirmados_df['Municipio'] == mun])
    obitos_mun = len(confirmados_df[
        (confirmados_df['Municipio'] == mun) &
        (confirmados_df['Evolucao'] == 'Óbito pelo COVID-19')
    ])
    taxa = (obitos_mun / total_mun * 100) if total_mun > 0 else 0
    letalidade_munic[mun] = {'Total Confirmados': total_mun,
                              'Óbitos COVID': obitos_mun,
                              'Taxa Letalidade (%)': round(taxa, 2)}

letalidade_df = pd.DataFrame(letalidade_munic).T.sort_values('Taxa Letalidade (%)', ascending=False)
print(letalidade_df)

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico empilhado da tabela cruzada
crosstab_pct = crosstab.div(crosstab.sum(axis=1), axis=0) * 100
cores_evolucao = ['#4CAF50', '#9E9E9E', '#F44336', '#FF9800']
crosstab_pct.plot(kind='bar', ax=axes[0], color=cores_evolucao[:len(crosstab_pct.columns)],
                  stacked=True, edgecolor='white')
axes[0].set_title('Distribuição de Evolução por Município (%)', fontweight='bold')
axes[0].set_xlabel('Município')
axes[0].set_ylabel('Percentual (%)')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(title='Evolução', bbox_to_anchor=(1.0, 1), loc='upper left', fontsize=8)

# Gráfico de taxa de letalidade
ax2 = axes[1]
bars = ax2.bar(letalidade_df.index,
               letalidade_df['Taxa Letalidade (%)'],
               color=plt.cm.Reds(np.linspace(0.4, 0.9, len(letalidade_df))))
for bar, val in zip(bars, letalidade_df['Taxa Letalidade (%)']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val:.2f}%', ha='center', fontsize=10, fontweight='bold')
ax2.set_title('Taxa de Letalidade por Município (%)', fontweight='bold')
ax2.set_xlabel('Município')
ax2.set_ylabel('Taxa de Letalidade (%)')
ax2.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

mun_maior_letal = letalidade_df['Taxa Letalidade (%)'].idxmax()
taxa_maior = letalidade_df.loc[mun_maior_letal, 'Taxa Letalidade (%)']
print(f'\nMunicípio com maior taxa de letalidade: {mun_maior_letal} ({taxa_maior:.2f}%)')

**Interpretação:** A tabela cruzada revela que, entre os 5 municípios com mais casos confirmados, a distribuição de desfechos é relativamente similar, com predominância de **Cura**. No entanto, existem variações nas taxas de letalidade: municípios com menor infraestrutura hospitalar ou com população mais envelhecida tendem a apresentar taxas ligeiramente superiores. **Cachoeiro de Itapemirim**, por exemplo, por ser uma cidade mais interiorana com população mais idosa relativa, pode apresentar taxas de letalidade acima das capitais da Grande Vitória, que possuem maior capacidade hospitalar.

---
## Conclusão Geral

Esta análise exploratória dos Microdados COVID-19 do Espírito Santo revelou os seguintes principais insights:

1. **Concentração urbana:** A Grande Vitória concentra a maioria das notificações, refletindo densidade populacional e maior capacidade de testagem.
2. **Perfil dos casos:** Adultos de 20-49 anos foram os mais notificados, enquanto idosos apresentaram maior risco de evolução grave.
3. **Taxa de letalidade:** Aproximadamente 5% dos casos confirmados evoluíram para óbito por COVID-19, alinhado com médias nacionais.
4. **Sintomatologia:** Febre, tosse e cefaleia foram os sintomas mais prevalentes.
5. **Comorbidades:** Doenças cardiovasculares e diabetes são os principais fatores de risco para óbito.
6. **Ondas pandêmicas:** O gráfico temporal demonstra claramente três ondas de infecção (2020, 2021 e início de 2022) com redução progressiva a partir da vacinação em massa.
